> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 11 · CONTEXT ENGINEERING</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Making your coding assistant write Sarvam code that runs</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Markdown docs · llms.txt · Context7 · Agent Skills — four layers, measured against the gotchas list</div>
</div>

**Time:** 65 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹4 &nbsp;·&nbsp; **Prereq:** Lab 00

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## The problem, in one sentence

Ask any coding assistant for a Sarvam snippet and it will confidently write
`client.chat.completions.create(...)` — a method that **does not exist** in this SDK.

It is not being stupid. It has read a hundred thousand OpenAI examples and roughly
none for Sarvam, so it pattern-matches to the thing it knows. The fix is not a better
model. **The fix is better context.**

This lab builds four layers of context, cheapest first, and then does something the
internet almost never does: **measures whether each one actually helped.**

| Layer | What it is | Setup cost | Freshness |
|---|---|---|---|
| 1 · **Markdown docs** | Append `.md` to any docs URL | Zero | Live |
| 2 · **llms.txt** | A curated index the vendor publishes for LLMs | Zero | Live |
| 3 · **Context7** | An MCP server that serves indexed docs on demand | One config block | Index lag |
| 4 · **Agent Skills** | Portable folders of SDK signatures + gotchas | One install | You own it |

> **This lab is cheap (≈ ₹4) and mostly offline.** Most cells fetch documents and
> read files. That makes it the best candidate in the series for a free campus
> session — high value, near-zero credit burn.

---
## 1 · The baseline — what a context-free assistant produces

We will use Sarvam-105B itself as the stand-in for "a coding assistant with no
special context". Same failure modes, and it keeps the lab self-contained.

In [3]:
TASK = (
    "Write a short Python snippet using the `sarvamai` SDK that:\n"
    "1. transcribes ./data/call.wav (Hindi, 8 kHz telephony audio), and\n"
    "2. sends the transcript to Sarvam's chat model for a one-line summary.\n"
    "Return ONLY code, no prose."
)

# ⚠️ Only added to the system message when context is actually passed — the
# LAYER 0 / no-context baseline below must stay a clean, unprompted control,
# or a later "did the instruction help" comparison has no fair floor to sit on.
CONTEXT_PRIORITY_INSTRUCTION = (
    "Reference documentation for this exact SDK is included below, before the "
    "task. Treat it as ground truth that overrides your own training — "
    "including patterns that look standard, idiomatic, or familiar from other "
    "APIs you have seen elsewhere. Where the documentation shows an exact code "
    "pattern (how a client is constructed, which parameters are required, "
    "optional, or must be omitted entirely), reproduce that pattern exactly. "
    "Do not add a parameter the documentation does not call for just because "
    "similar APIs usually take one."
)

def generate(task, context="", model="sarvam-105b", label=""):
    """Ask the model for code, optionally with retrieved context prepended."""
    if context:
        sys_msg = ("You are a senior Python engineer. Output only runnable code. "
                    + CONTEXT_PRIORITY_INSTRUCTION)
        user = f"{context}\n\n---\n\n{task}"
    else:
        sys_msg = "You are a senior Python engineer. Output only runnable code."
        user = task
    r = client.chat.completions(
        model=model,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user",   "content": user}],
        max_tokens=900, temperature=0.1, reasoning_effort=None,
    )
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    out = r.choices[0].message.content or ""
    if label:
        print(f"── {label} ── ({r.usage.prompt_tokens} in / {r.usage.completion_tokens} out)")
    return out

baseline = generate(TASK, label="LAYER 0 · no context")
print(baseline[:])

── LAYER 0 · no context ── (94 in / 203 out)

```python
from sarvamai import SpeechClient, ChatClient
from sarvamai.models import TranscriptRequest

# 1. Transcribe
speech = SpeechClient()
with open("./data/call.wav", "rb") as f:
    transcript = speech.transcribe(
        file=f,
        language="hi-IN",
        audio_format="wav",
        sample_rate_hz=8000,
        model="saaras:v3"
    )
text = transcript.transcript

# 2. Summarize
chat = ChatClient()
summary = chat.chat(
    messages=[
        {"role": "user", "content": f"Summarize this Hindi transcript in one line: {text}"}
    ],
    model="sarvam-1",
    temperature=0.3
)
print(summary.choices[0].message.content)
```


In [4]:
# Paste the above Python code for verification

# import asyncio
# from sarvamai import SarvamAIClient

# async def main():
#     client = SarvamAIClient()
    
#     # 1. Transcribe Hindi 8kHz telephony audio
#     transcript = await client.speech.transcribe(
#         audio_path="./data/call.wav",
#         language="hi-IN",
#         sample_rate=8000,
#         encoding="linear16",
#         model="saarika",
#         task="transcribe"
#     )
    
#     # 2. Send transcript to chat model for one-line summary
#     response = await client.chat.generate(
#         messages=[
#             {"role": "user", "content": f"Summarize the following Hindi transcript in one line:\n{transcript}"}
#         ],
#         model="sarvam-m",
#         temperature=0.2
#     )
    
#     print(response["choices"][0]["message"]["content"])

# if __name__ == "__main__":
#     asyncio.run(main())

# Trial - 2    

# import os
# from sarvamai import SarvamAI

# api_key = API_KEY     #os.environ.get("SARVAM_API_KEY")
# if not api_key:
#     raise ValueError("Set SARVAM_API_KEY environment variable")

# client = SarvamAI(api_key=api_key)

# # 1. Transcribe Hindi telephony audio
# audio_path = "./data/call.wav"
# with open(audio_path, "rb") as f:
#     transcript = client.speech_to_text(
#         audio=f,
#         language="hi-IN",
#         sample_rate=8000,
#         model="saarika:v1",
#     )
# text = transcript["transcript"]

# # 2. Summarize with chat model
# summary = client.chat(
#     model="sarvam-1",
#     messages=[{"role": "user", "content": f"Summarize this in one line:\n{text}"}],
# )
# print(summary["choices"][0]["message"]["content"])

# Trial - 1
    
# from sarvamai import Audio, Chat
# import os

# audio_path = os.path.expanduser("./data/call.wav")
# audio_bytes = open(audio_path, "rb").read()

# transcription = Audio.transcribe(
#     file=audio_bytes,
#     language="hi",
#     sample_rate=8000,
#     model="saaras:v3"
# )

# response = Chat.chat(
#     messages=[{"role": "user", "content": transcription["transcript"]}],
#     model="sarvam-m"
# )

# print(response["choices"][0]["message"]["content"])

---
## 2 · The rubric — your own gotchas list, as code

Deck 08 slide 4 is the most-screenshotted artefact of the whole workshop. Here it
becomes an **automated grader**. Each gotcha is a regex for the *wrong* pattern; a
generation scores a point for every trap it avoids.

Deterministic, free, and transparent — no judge model, no API call, no opinion.

In [5]:
import re

# Each rule: (id, human description, regex matching the WRONG pattern, the fix)
GOTCHAS = [
    ("no-create",   "client.chat.completions.create(...) — method does not exist",
     r"chat\s*\.\s*completions\s*\.\s*create\s*\(",
     "client.chat.completions(...) — no .create"),

    ("rest-sample-rate", "sample_rate= passed to the REST transcribe() call",
     r"speech_to_text\s*\.\s*transcribe\s*\([^)]*sample_rate\s*=",
     "Not a REST parameter — the WAV header carries it (Lab 02 §2)"),

    ("doc-language-code", "language_code= on a Document AI call",
     # [\s\S] so the rule still fires when the call spans several lines
     r"(doc_ai|document_intelligence)[\s\S]{0,160}?language_code\s*=",
     "Use language= on Document AI"),

    ("md-not-markdown", 'output_format="markdown" instead of "md"',
     r"output_format\s*=\s*[\"']markdown[\"']",
     'Use "md"'),

    ("schema-dict",  "schema passed as a dict instead of a JSON string",
     r"schema\s*=\s*\{",
     "json.dumps(schema)"),

    ("bare-file",    "file=open(...) on a Document AI call",
     r"(doc_ai|document_intelligence)[\s\S]{0,160}?file\s*=\s*open\s*\(",
     "file=[(name, handle, mime)] — an array"),

    ("terminal-state", "reading results without polling for a terminal state",
     r"get_results?\s*\((?![^)]*status)",
     "Poll get_status() / wait_until_complete() first"),

    ("implicit-model", "no explicit model= argument anywhere",
     r"\A(?:(?!model\s*=).)*\Z",
     "Always pass model= explicitly — defaults drift"),

# ── Added after a live run: baseline code scored 8/8 despite being
    # thoroughly broken. None of the 8 rules above were written for THESE
    # mistakes — a rubric only catches what it was told to look for.

    ("wrong-client-kwarg", "SarvamAI(api_key=...) instead of api_subscription_key=",
     r"SarvamAI\s*\(\s*api_key\s*=",
     "SarvamAI(api_subscription_key=...)"),

    ("bare-stt-call", "client.speech_to_text(...) called directly, skipping .transcribe",
     r"client\s*\.\s*speech_to_text\s*\(",
     "client.speech_to_text.transcribe(file=..., ...)"),

    ("bare-chat-call", "client.chat(...) called directly, skipping .completions",
     r"client\s*\.\s*chat\s*\(",
     "client.chat.completions(...) — no .create"),

    ("invalid-model-name", "a model name that does not exist on this platform",
     r"model\s*=\s*[\"'](saarika:v1|sarvam-1)[\"']",
     "saaras:v3 for STT; sarvam-105b / sarvam-30b / sarvam-m for chat"),

    ("dict-response-access", "SDK response treated as a dict instead of an object",
     r"\[[\"']transcript[\"']\]|\[[\"']choices[\"']\]\s*\[\s*0\s*\]\s*\[[\"']message[\"']\]\s*\[[\"']content[\"']\]",
     "response.transcript / response.choices[0].message.content — attribute access"),

    ("nonexistent-class", "importing Audio/Chat classes that don't exist in the SDK",
     r"from\s+sarvamai\s+import[^\n]*\b(Audio|Chat)\b",
     "Only SarvamAI / AsyncSarvamAI are exported at the top level"),

   # ── Added after another live run: SarvamAI() with empty parens is not a
    # hard crash — the constructor's api_subscription_key default is
    # os.getenv("SARVAM_API_KEY"), so it silently works IF that env var
    # happens to be set. The gotcha isn't that it's broken; it's that a
    # standalone script has no business assuming a var it never checked is
    # set. Every other client construction in this workshop reads the key
    # into a variable and asserts on it before use — that's the pattern
    # this rule is actually enforcing, not "always pass the kwarg".
    ("silent-env-key", "SarvamAI() called bare, relying on an unchecked env var",
     r"SarvamAI\s*\(\s*\)",
     'API_KEY = os.environ.get("SARVAM_API_KEY"); assert API_KEY, "Set SARVAM_API_KEY"; '
     "client = SarvamAI(api_subscription_key=API_KEY) — validate before you depend on it"),

    # ── Added after TWO independent live runs, both against the same fetched
    # STT page: the model invented input_audio_codec="amr_wb", then on a
    # rerun invented "amr_nb" — different guesses, same root cause. Neither
    # value exists in the SDK's actual enum (only bare "amr" does), and the
    # fetched doc says the parameter is unnecessary for a WAV file — it's
    # auto-detected, and only required for raw PCM. This isn't a "context
    # missing" failure; the doc said not to. It's the task's own wording
    # ("8 kHz telephony audio") priming a plausible-sounding parameter the
    # model adds unprompted — a different failure mode than misreading or
    # ignoring a shown pattern, and one a "trust the context" instruction
    # alone does not reliably suppress.
    ("hallucinated-audio-codec", 'input_audio_codec set to a codec variant that does not exist ("amr_wb"/"amr_nb" — only "amr" is real)',
     r"input_audio_codec\s*=\s*[\"']amr_(wb|nb)[\"']",
     "Drop input_audio_codec for a WAV file — it's auto-detected; only pcm_s16le/pcm_l16/pcm_raw need it"),                                   
]

def score(codeblock, verbose=False):
    """Return (points, max, [failed ids]). One point per gotcha AVOIDED."""
    failed = []
    for gid, desc, pattern, fix in GOTCHAS:
        if re.search(pattern, codeblock, flags=re.S):
            failed.append(gid)
            if verbose:
                print(f"   ✕ {gid:<18} {desc}")
                print(f"     fix: {fix}")
    return len(GOTCHAS) - len(failed), len(GOTCHAS), failed

pts, mx, failed = score(baseline, verbose=True)
print(f"\nBASELINE SCORE: {pts}/{mx} gotchas avoided")

   ✕ invalid-model-name a model name that does not exist on this platform
     fix: saaras:v3 for STT; sarvam-105b / sarvam-30b / sarvam-m for chat

BASELINE SCORE: 15/16 gotchas avoided


> **A note on rule accuracy — and why this lab corrects the deck.** The original
> Deck 08 table listed *"8 kHz audio, no `sample_rate` → garbage transcript"*. Lab 02
> disproved that: `sample_rate` is **not a REST parameter at all**, and passing it
> raises `TypeError`. The rule above encodes the *corrected* fact. If your rubric
> encodes a stale belief, it will punish correct code — which is exactly the failure
> mode this whole lab exists to prevent.

---
## 3 · Layer 1 — Markdown docs, for free

Every page on `docs.sarvam.ai` is available as clean Markdown: **append `.md` to the
URL**. No key, no SDK, no setup.

Why it matters: HTML spends most of its tokens on navigation, scripts and styling.
Markdown is nearly all content. Measure the difference.

In [6]:
import httpx

PAGE = "https://docs.sarvam.ai/api-reference/chat/chat-completions"

def fetch(url, timeout=30):
    try:
        r = httpx.get(url, timeout=timeout, follow_redirects=True)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  fetch failed {url}: {type(e).__name__}: {e}")
        return ""

html = fetch(PAGE)
mark = fetch(PAGE + ".md")

def approx_tokens(s): return len(s) // 4

print(f"{'form':<12}{'chars':>10}{'~tokens':>10}")
print("─" * 32)
print(f"{'HTML':<12}{len(html):>10,}{approx_tokens(html):>10,}")
print(f"{'Markdown':<12}{len(mark):>10,}{approx_tokens(mark):>10,}")
if html and mark:
    print(f"\nMarkdown is {len(html)/max(len(mark),1):.1f}x smaller — "
          f"~{approx_tokens(html)-approx_tokens(mark):,} tokens saved per page")

form             chars   ~tokens
────────────────────────────────
HTML         1,203,553   300,888
Markdown        15,547     3,886

Markdown is 77.4x smaller — ~297,002 tokens saved per page


In [7]:
print(mark[:900] if mark else "(markdown fetch failed — check network access)")

> For clean Markdown of any page, append `.md` to the page URL.
> For a complete documentation index, see https://docs.sarvam.ai/llms.txt.
> For full documentation content in one file, see https://docs.sarvam.ai/llms-full.txt.
> For AI client integration (Claude Code, Cursor, etc.), connect to the MCP server at https://docs.sarvam.ai/_mcp/server.

# Chat Completion

POST https://api.sarvam.ai/v1/chat/completions
Content-Type: application/json

Creates a model response for the given chat conversation. This endpoint serves **only** `sarvam-105b` and `sarvam-105b-conversations`.

Reference: https://docs.sarvam.ai/api-reference/chat/chat-completions

## Authentication

- `Authorization` header (bearer token, required)
- `api-subscription-key` header (required)

## Request

### Headers

- `api-subscription-key` (string, optional, nullable) — API subscription key in sk_xxx format. [Steps to ge


In [8]:
# ── Does fetching BOTH relevant pages (not just one) close more gotchas? ──
# The task needs two capabilities: speech-to-text AND chat. The single page
# fetched above only documents chat — it structurally cannot fix an STT
# mistake, no matter how good the model is. Fetch the STT reference page too
# and see whether covering both halves of the task closes more of the rubric.
STT_PAGE = "https://docs.sarvam.ai/api-reference/speech-to-text/transcribe"
mark_stt = fetch(STT_PAGE + ".md")

ctx_two_pages = f"# Chat completions\n{mark}\n\n# Speech to text\n{mark_stt}" if (mark and mark_stt) else ""
print(f"chat page  : ~{approx_tokens(mark):,} tokens")
print(f"stt page   : ~{approx_tokens(mark_stt):,} tokens")
print(f"combined   : ~{approx_tokens(ctx_two_pages):,} tokens")

chat page  : ~3,886 tokens
stt page   : ~3,114 tokens
combined   : ~7,010 tokens


In [9]:
# ── Generate against this combined context, right now — don't wait for §7 ──
# The whole point of fetching the STT page too was to see whether covering
# both halves of the task closes gotchas the single chat page structurally
# can't touch. Section 7 folds this into a six-way table later, but that's
# a rerun under a different label — check the actual effect here first.
if ctx_two_pages:
    two_page_out = generate(TASK, ctx_two_pages, label="LAYER 1b · markdown (chat+stt)")
    print(two_page_out[:])
    pts, mx, failed = score(two_page_out, verbose=True)
    base_pts = score(baseline)[0]
    print(f"\nSCORE: {pts}/{mx} gotchas avoided  (baseline with no context: {base_pts}/{mx})")
else:
    print("(context unavailable — check network access)")

── LAYER 1b · markdown (chat+stt) ── (9129 in / 213 out)

```python
from sarvamai import SarvamAI
from pathlib import Path

client = SarvamAI(
    api_subscription_key="YOUR_API_KEY_HERE",
)

# Transcribe Hindi, 8 kHz telephony audio
transcript = client.speech_to_text.transcribe(
    file=Path("./data/call.wav"),
    language_code="hi-IN",
    model="saaras:v3",
    mode="transcribe",
    input_audio_codec="amr-wb",
)

# Send transcript to chat model for one-line summary
response = client.chat.completions(
    messages=[
        {
            "role": "user",
            "content": f"Summarize this transcript in one line: {transcript}",
        }
    ],
    model="sarvam-105b",
)

print(response.choices[0].message.content)
```

SCORE: 16/16 gotchas avoided  (baseline with no context: 15/16)


In [10]:
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key = API_KEY)  # change 1 needed

# Transcribe Hindi, 8 kHz telephony audio
transcription = client.speech_to_text.transcribe(
    file=open( "./data/call_8k.wav", 'rb'), #   "./data/call_8k.wav", change 2 needed
    language_code="hi-IN",
    model="saaras:v4",
    mode="transcribe",
    input_audio_codec="wav",     #"amr_wb",        # change 3  needed
)  

# Send transcript to chat model for a one-line summary
response = client.chat.completions(
    messages=[
        {"role": "user", "content": f"Summarize the following Hindi transcript in one line:\n\n{transcription.transcript}"}
    ],
    model="sarvam-105b",
    temperature=0.3,  
    max_tokens=1000,          # change 4 needed   
)

print(response.choices[0].message.content)

None


In [11]:
transcription

SpeechToTextResponse(request_id='20260828_78c2ed87-2d48-4492-8317-e8a651cbb982', transcript='नमस्ते, मैं अपने लोन के बारे में जानकारी चाहता हूँ।', timestamps=None, language_code='hi-IN', language_probability=None)

---
## 4 · Layer 2 — `llms.txt`, the vendor's own curated index

[`llms.txt`](https://llmstxt.org/) is a small open standard. The format is strict and
deliberately tiny:

| Part | Required | Content |
|---|---|---|
| H1 | **Yes** — the only required part | Project name |
| Blockquote | No | One-paragraph summary |
| Body | No | Any markdown *except headings* |
| H2 sections | No | Lists of `[name](url): notes` links |

**One honest correction to what you will read elsewhere:** `llms-full.txt` is **not**
part of the spec. It is a widely-adopted convention for "everything, concatenated",
and Sarvam ships one — but do not expect every vendor to.

In [12]:
LLMS      = "https://docs.sarvam.ai/llms.txt"
LLMS_FULL = "https://docs.sarvam.ai/llms-full.txt"

idx  = fetch(LLMS)
full = fetch(LLMS_FULL, timeout=60)

print(f"{'file':<16}{'chars':>12}{'~tokens':>10}")
print("─" * 38)
print(f"{'llms.txt':<16}{len(idx):>12,}{approx_tokens(idx):>10,}")
print(f"{'llms-full.txt':<16}{len(full):>12,}{approx_tokens(full):>10,}")
print()
print("llms.txt is the MAP. llms-full.txt is the TERRITORY.")
print("Feed the map when you need to choose; feed the territory when you need detail.")

file                   chars   ~tokens
──────────────────────────────────────
llms.txt              13,642     3,410
llms-full.txt         13,642     3,410

llms.txt is the MAP. llms-full.txt is the TERRITORY.
Feed the map when you need to choose; feed the territory when you need detail.


In [13]:
# Parse llms.txt per the spec: H1, optional blockquote, H2 link sections
def parse_llms_txt(text):
    title, summary, sections, current = None, [], {}, None
    for line in text.split("\n"):
        s = line.strip()
        if s.startswith("# ") and title is None:
            title = s[2:].strip()
        elif s.startswith("> "):
            summary.append(s[2:].strip())
        elif s.startswith("## "):
            current = s[3:].strip(); sections[current] = []
        elif s.startswith("-") and current:
            m = re.match(r"-\s*\[([^\]]+)\]\(([^)]+)\)\s*:?\s*(.*)", s)
            if m:
                sections[current].append({"name": m.group(1), "url": m.group(2),
                                          "note": m.group(3)})
    return title, " ".join(summary), sections

if idx:
    title, summary, sections = parse_llms_txt(idx)
    print("TITLE  :", title)
    print("SUMMARY:", (summary or "(none)")[:200])
    print(f"\n{len(sections)} section(s):")
    for name, links in sections.items():
        print(f"  {name:<34} {len(links):>3} links")
        for l in links[:2]:
            print(f"      {l['name'][:40]:<42} {l['url'][:52]}")
else:
    title, summary, sections = None, "", {}
    print("(llms.txt fetch failed)")

TITLE  : Sarvam AI Developer Documentation
SUMMARY: Build with Sarvam AI. Speech-to-text, text-to-speech, translation, chat, and document intelligence APIs for 22 Indian languages, plus no-code platforms for voice, work, coding, and content agents.

2 section(s):
  Instructions for AI Agents           0 links
  Products                            38 links
      Model APIs                                 https://docs.sarvam.ai/api/llms.txt
      Welcome                                    https://docs.sarvam.ai/api/getting-started/welcome.m


In [14]:
# Use the index the way an agent would: pick the relevant pages, fetch only those
def retrieve(query_words, sections, limit=2):
    """Naive relevance: score links by keyword overlap, fetch the winners as .md"""
    scored = []
    for name, links in sections.items():
        for l in links:
            blob = f"{name} {l['name']} {l['note']}".lower()
            hits = sum(w.lower() in blob for w in query_words)
            if hits: scored.append((hits, l))
    scored.sort(key=lambda x: -x[0])
    out = []
    for _, l in scored[:limit]:
        url = l["url"]
        if not url.startswith("http"):
            url = "https://docs.sarvam.ai" + url
        body = fetch(url if url.endswith(".md") else url + ".md")
        if body:
            out.append(f"# {l['name']}\n{body[:6000]}")
            print(f"  retrieved {l['name']}  ({approx_tokens(body):,} tokens)")
    return "\n\n".join(out)

ctx_llms = retrieve(["speech", "text", "transcribe", "chat"], sections) if sections else ""
print(f"\nretrieved context: ~{approx_tokens(ctx_llms):,} tokens")

  retrieved Model APIs  (67 tokens)
  retrieved Generating Speech  (1,881 tokens)

retrieved context: ~1,576 tokens


---
## 5 · Layer 3 — Context7, docs on demand over MCP

[Context7](https://context7.com/) is a documentation index that coding assistants
query **through MCP**, so the docs arrive as tool results rather than as something you
paste. Sarvam maintains a page for it, and the docs are **already indexed** —
[`docs.sarvam.ai/api/developer-tools/context7`](https://docs.sarvam.ai/api/developer-tools/context7).

**The Sarvam library ID is `/websites/sarvam_ai`.**

### Setup — one config block, no API key

```json
{
  "mcpServers": {
    "context7": {
      "command": "npx",
      "args": ["-y", "@upstash/context7-mcp"]
    }
  }
}
```

Drop that into Claude Code, Cursor, Windsurf or any MCP client. A free key from
`context7.com/dashboard` raises rate limits but is not required.

### Using it

Two tools: **`resolve-library-id`** (name → Context7 ID) and **`query-docs`** (ID +
question → docs). In practice you skip the resolve step by naming the library:

> *"Use library `/websites/sarvam_ai`. Write a Python script that transcribes
> `audio.wav` with the latest Sarvam speech-to-text model."*

### When Context7 is the **wrong** tool

Be honest about this in the room:

- **Index lag.** Context7 re-crawls on its own schedule. For an API shipping as fast
  as Sarvam's, the vendor's own `llms.txt` is fresher by definition.
- **Another moving part.** One more MCP server, one more failure mode, one more thing
  to explain to a new team member.
- **You already have the docs.** If a page is two fetches away, layer 1 is simpler.

Context7 earns its place when you are working across **many** libraries and want one
uniform way to reach all their docs — not when you need the freshest possible truth
about one.

In [15]:
# Verify the config is well-formed, and (optionally) that the server starts.
import json, shutil, subprocess

CONTEXT7_CONFIG = {                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
    "mcpServers": {
        "context7": {"command": "npx", "args": ["-y", "@upstash/context7-mcp"]}
    }
}
print(json.dumps(CONTEXT7_CONFIG, indent=2))
Path("context7_mcp_config.json").write_text(json.dumps(CONTEXT7_CONFIG, indent=2))
print("\nwrote context7_mcp_config.json — paste into your MCP client\n")

SARVAM_LIBRARY_ID = "/websites/sarvam_ai"
print("Sarvam library ID:", SARVAM_LIBRARY_ID)
print("Browse it at    : https://context7.com/websites/sarvam_ai")
    
if shutil.which("npx") is None:
    print("\n(npx not found — install Node.js to run the Context7 server locally)")
else:
    print("\n(npx present — the config above will work in your MCP client)")

{
  "mcpServers": {
    "context7": {
      "command": "npx",
      "args": [
        "-y",
        "@upstash/context7-mcp"
      ]
    }
  }
}

wrote context7_mcp_config.json — paste into your MCP client

Sarvam library ID: /websites/sarvam_ai
Browse it at    : https://context7.com/websites/sarvam_ai

(npx not found — install Node.js to run the Context7 server locally)


---
## 6 · Layer 4 — Agent Skills, the layer you own

The first three layers hand your assistant **documentation**. Documentation tells it
what the API *offers*. It does not tell it what people *get wrong*.

An [Agent Skill](https://agentskills.io/specification) is a folder your assistant
loads that carries exactly that: signatures, and the traps. Sarvam publishes five —
`chat`, `speech-to-text`, `text-to-speech`, `translate`, `voice-agents` — and they
point at `llms.txt` for anything deeper.

```bash
npx skills add sarvamai/skills            # all five
npx skills add sarvamai/skills --skill chat
npx skills add sarvamai/skills --list     # browse first
```

Works in Claude Code, Cursor, Windsurf, and anything implementing the spec.

### The spec, exactly

A skill is a directory with a `SKILL.md` at minimum:

```
skill-name/
├── SKILL.md          # required: YAML frontmatter + markdown body
├── scripts/          # optional: runnable code
├── references/       # optional: detail loaded on demand
└── assets/           # optional: templates, data
```

| Frontmatter | Required | Constraint |
|---|---|---|
| `name` | **Yes** | ≤64 chars, `a-z0-9-` only, no leading/trailing/double hyphen, **must match the directory name** |
| `description` | **Yes** | ≤1024 chars. Say *what it does* **and** *when to use it* |
| `license` | No | Name or bundled file |
| `compatibility` | No | ≤500 chars — environment needs |
| `metadata` | No | Arbitrary string→string map |
| `allowed-tools` | No | Space-separated pre-approved tools (experimental) |

**Progressive disclosure is the design constraint that matters.** The agent loads:

1. `name` + `description` (~100 tokens) — for **every** installed skill, at startup
2. The `SKILL.md` body (**keep under 5,000 tokens / 500 lines**) — only when activated
3. `references/`, `scripts/`, `assets/` — only when actually needed

So the description is doing triage for a model that has not read your skill yet.
Write it for that job.

In [16]:
# ── Author a real skill: teach an assistant to always meter Sarvam calls ──
SKILL_DIR = Path("skills/sarvam-cost-metering")
(SKILL_DIR / "references").mkdir(parents=True, exist_ok=True)

SKILL_MD = '''---
name: sarvam-cost-metering
description: Adds rupee cost tracking to Sarvam AI API calls. Use whenever writing or reviewing Python that calls the sarvamai SDK - speech_to_text, text_to_speech, chat.completions, translate, or doc_ai - so that every billed call is accompanied by a CostMeter entry and the script prints a total in rupees.
license: Apache-2.0
compatibility: Requires Python 3.10+ and the sarvamai SDK
metadata:
  author: AIVidhya4Sarvam
  version: "1.0"
---

# Sarvam cost metering

Every Sarvam API call costs money. Code that calls Sarvam without tracking spend is
incomplete. When you write or review such code, attach a meter entry to every billed
call and print a total at the end.

## The rule

For each billed call, add the matching meter line immediately after it:

| API | Billed by | Meter call |
|---|---|---|
| `speech_to_text.transcribe` | audio seconds | `cost.stt(seconds)` |
| `speech_to_text` + diarization | audio seconds | `cost.stt(seconds, diarized=True)` |
| `text_to_speech.convert` | characters | `cost.tts(len(text), v3=False)` |
| `text.translate` / `transliterate` | characters | `cost.text(len(s), kind="translate")` |
| `chat.completions` | tokens | `cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)` |
| `doc_ai` extract/digitise | pages | `cost.doc(n_pages)` |

End every script with `cost.report()`.

## Correct usage

```python
from cost_meter import CostMeter
cost = CostMeter()

r = client.chat.completions(model="sarvam-105b", messages=msgs,
                            max_tokens=400, reasoning_effort=None)
cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)

cost.report()
```

## Non-negotiables

- Always pass `model=` explicitly. Defaults drift between releases.
- `client.chat.completions(...)` has **no** `.create()`. Writing
  `client.chat.completions.create(...)` raises `AttributeError`.
- If `reasoning_effort` is left on with a small `max_tokens`, reasoning consumes the
  whole budget and `content` comes back `None`. Pass `reasoning_effort=None` or raise
  `max_tokens`.
- MCP tool calls do not report usage back. Meter them from your own inputs.

See [references/RATES.md](references/RATES.md) for current pricing.
'''

RATES_MD = '''# Sarvam rates (August 2026)

Verify at https://docs.sarvam.ai/api/getting-started/pricing before quoting.

| Item | Rate |
|---|---|
| STT | Rs 30 / hour |
| STT + diarization | Rs 45 / hour |
| TTS bulbul:v2 | Rs 15 / 10k chars |
| TTS bulbul:v3 | Rs 30 / 10k chars |
| Translate / transliterate | Rs 20 / 10k chars |
| LLM input | Rs 29.28 / 1M tokens |
| LLM cached input | Rs 10.98 / 1M tokens |
| LLM output | Rs 73.20 / 1M tokens |
| Document AI | Rs 0.50 / page |

Estimates are a conservative upper bound: prompt caching, free-tier credit and
invoice rounding all push the real bill slightly lower.
'''

(SKILL_DIR / "SKILL.md").write_text(SKILL_MD, encoding="utf-8")
(SKILL_DIR / "references" / "RATES.md").write_text(RATES_MD, encoding="utf-8")

for p in sorted(SKILL_DIR.rglob("*")):
    print(f"  {p.relative_to(SKILL_DIR.parent)}  ({p.stat().st_size if p.is_file() else '-'} bytes)")

  sarvam-cost-metering/SKILL.md  (2201 bytes)
  sarvam-cost-metering/references  (- bytes)
  sarvam-cost-metering/references/RATES.md  (608 bytes)


In [17]:
# ── Validate against the spec, before any agent ever sees it ─────────────
def validate_skill(skill_dir):
    errs, warns = [], []
    p = Path(skill_dir); md_path = p / "SKILL.md"
    if not md_path.exists():
        return ["SKILL.md is missing"], []

    text = md_path.read_text(encoding="utf-8")
    if not text.startswith("---"):
        return ["SKILL.md must begin with YAML frontmatter (---)"], []

    fm_raw = text.split("---", 2)[1]
    fm = {}
    for line in fm_raw.split("\n"):
        if ":" in line and not line.startswith((" ", "\t", "-")):
            k, v = line.split(":", 1)
            fm[k.strip()] = v.strip()

    name = fm.get("name", "")
    if not name:                                errs.append("`name` is required")
    if len(name) > 64:                          errs.append("`name` exceeds 64 chars")
    if not re.fullmatch(r"[a-z0-9]+(-[a-z0-9]+)*", name or ""):
        errs.append(f"`name` must be lowercase a-z0-9 with single hyphens: {name!r}")
    if name != p.name:
        errs.append(f"`name` ({name!r}) must match directory ({p.name!r})")

    desc = fm.get("description", "")
    if not desc:                                errs.append("`description` is required")
    if len(desc) > 1024:                        errs.append("`description` exceeds 1024 chars")
    if desc and " use " not in desc.lower() and not desc.lower().startswith("use "):
        warns.append("`description` should say WHEN to use the skill, not just what it does")

    comp = fm.get("compatibility", "")
    if len(comp) > 500:                         errs.append("`compatibility` exceeds 500 chars")

    body = text.split("---", 2)[2]
    n_lines, n_tok = len(body.split("\n")), len(body) // 4
    if n_lines > 500:   warns.append(f"body is {n_lines} lines — spec recommends under 500")
    if n_tok   > 5000:  warns.append(f"body is ~{n_tok} tokens — spec recommends under 5000")

    print(f"  name        : {name}")
    print(f"  description : {len(desc)} chars")
    print(f"  body        : {n_lines} lines, ~{n_tok} tokens")
    return errs, warns

errs, warns = validate_skill(SKILL_DIR)
print()
for e in errs:  print("  ERROR  ", e)
for w in warns: print("  WARN   ", w)
print("\n" + ("✓ VALID — conforms to the Agent Skills spec" if not errs
               else f"✕ {len(errs)} error(s) to fix"))
print("\nOfficial validator:  skills-ref validate ./skills/sarvam-cost-metering")

  name        : sarvam-cost-metering
  description : 295 chars
  body        : 48 lines, ~431 tokens


✓ VALID — conforms to the Agent Skills spec

Official validator:  skills-ref validate ./skills/sarvam-cost-metering


### A second skill — scoped to cover the rubric, not just cost

`sarvam-cost-metering` above is deliberately narrow: it teaches metering discipline,
and only incidentally fixes 1-2 gotchas as a side effect. It was never meant to be a
comprehensive "avoid every SDK mistake" reference — so it's unfair to expect it to
drive §7's score to a clean sweep.

This second skill is scoped differently: it exists specifically to cover every rule
currently in `GOTCHAS`. That's not circular in a bad way — it's the actual mechanism
Agent Skills are for. You choose what a skill covers; a fetched doc page does not.

In [18]:
# ── Author sarvam-sdk-gotchas: scoped to close every rule in GOTCHAS ──────
GOTCHAS_SKILL_DIR = Path("skills/sarvam-sdk-gotchas")
GOTCHAS_SKILL_DIR.mkdir(parents=True, exist_ok=True)

GOTCHAS_SKILL_MD = '''---
name: sarvam-sdk-gotchas
description: Prevents the most common sarvamai Python SDK mistakes - wrong client constructor kwarg, constructing the client with no key validation, calling speech_to_text or chat as bare methods instead of .transcribe/.completions, invalid model names, inventing audio codec parameters that do not exist, treating SDK responses as dicts instead of objects, non-existent imports, and Document AI parameter/schema mistakes. Use whenever writing, reviewing, or debugging Python code that imports or calls the sarvamai SDK.
license: Apache-2.0
compatibility: Requires Python 3.10+ and the sarvamai SDK
metadata:
  author: AIVidhya4Sarvam
  version: "1.0"
---

# Sarvam SDK gotchas

The sarvamai SDK looks like the OpenAI SDK in places and diverges sharply in others.
Every mistake below has actually been generated by an LLM with no Sarvam context.

## Client construction

```python
from sarvamai import SarvamAI
client = SarvamAI(api_subscription_key="...")   # NOT api_key=
```

`api_key=` raises `TypeError: unexpected keyword argument`. The constructor kwarg and
the HTTP header are both `api_subscription_key` — never `api_key` or `Authorization: Bearer`.

`SarvamAI()` with empty parens does not crash by itself — the constructor defaults
`api_subscription_key` to `os.getenv("SARVAM_API_KEY")`, so it silently works if that
env var happens to be set. That is not a reason to write it: a standalone script
should never depend on an env var it never checked. Read the key into a variable and
assert on it before constructing the client:

```python
API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, "Set SARVAM_API_KEY"
client = SarvamAI(api_subscription_key=API_KEY)
```

## Chat completions

```python
r = client.chat.completions(model="sarvam-105b", messages=[...], max_tokens=400,
                            reasoning_effort=None)
print(r.choices[0].message.content)   # attribute access, not r["choices"][0]...
```

- There is **no `.create()`**. `client.chat.completions.create(...)` raises
  `AttributeError`. `client.chat(...)` (skipping `.completions` entirely) does not
  exist either.
- Valid chat models: `sarvam-105b`, `sarvam-30b`, `sarvam-m` (being phased out —
  prefer `sarvam-105b`/`sarvam-30b`). `sarvam-1` is not a real model.
- `reasoning_effort` defaults to `"low"` (reasoning ON). A small `max_tokens` lets
  reasoning consume the whole budget and `content` comes back `None`. Pass
  `reasoning_effort=None` explicitly, or raise `max_tokens`.

## Speech-to-text

```python
with open(path, "rb") as f:
    r = client.speech_to_text.transcribe(file=f, model="saaras:v3",
                                         language_code="hi-IN", mode="transcribe")
print(r.transcript)   # attribute access, not r["transcript"]
```

- `client.speech_to_text(...)` (bare call, skipping `.transcribe`) does not exist.
- Valid STT models: `saaras:v3` (default), `saaras:v4`. `saarika:v1` is not real.
- `sample_rate=` is **not a REST parameter** — the WAV header carries it; passing it
  raises `TypeError`. It only applies on the streaming path (raw samples, no header).
- REST audio is capped at 30s; use the Batch API or WebSocket streaming beyond that.
- **Do not add `input_audio_codec=` for a WAV, MP3, or any container format the API
  already recognizes — it is auto-detected.** This parameter exists ONLY for raw PCM
  (`pcm_s16le`, `pcm_l16`, `pcm_raw`), which has no header to detect from. There is no
  `"amr_wb"` or `"amr_nb"` value in the SDK — those are invented; the only valid `amr`
  value is bare `"amr"`, and even that should not be added to a WAV file just because
  the audio happens to be low-bitrate telephony-style. If the task description says
  "telephony audio," that is not a cue to add a codec parameter.

## Response objects are objects, not dicts

Every SDK response is a Pydantic-style object — use attribute access:

```python
transcript = response.transcript              # not response["transcript"]
content    = r.choices[0].message.content     # not r["choices"][0]["message"]["content"]
```

## Imports

Only `SarvamAI` and `AsyncSarvamAI` are exported at the top level:

```python
from sarvamai import SarvamAI, AsyncSarvamAI
```

There is no `Audio`, `Chat`, `Translate`, or similar per-capability class.
`Audio.transcribe(...)` / `Chat.chat(...)` are hallucinated — always go through a
`SarvamAI` client instance.

## Document AI

- Use `language=`, not `language_code=`.
- `output_format="md"`, not `"markdown"` — the latter returns 400.
- `schema=` must be a **JSON string**: `schema=json.dumps(schema_dict)`, never a dict.
- `file=` is an **array of tuples**: `file=[(name, handle, mime)]`, not a bare `open(...)` handle.
- The extract schema does not support a `required` key — `SCHEMA_INVALID`.
- Always poll `get_status()` / `wait_until_complete()` before `get_results()`.
  `partially_completed` is a terminal state and is easy to miss.

## The one rule that covers most of the above

Always pass `model=` explicitly, and use the exact model names in this file — never
guess or reuse an OpenAI model name. Defaults and names drift between releases
faster than any cached knowledge. The same discipline applies to every other
parameter: if the documentation does not show it, do not add it — a plausible-looking
parameter borrowed from a similar API or from the task's own wording is still a guess.
'''

(GOTCHAS_SKILL_DIR / "SKILL.md").write_text(GOTCHAS_SKILL_MD, encoding="utf-8")
for p in sorted(GOTCHAS_SKILL_DIR.rglob("*")):
    print(f"  {p.relative_to(GOTCHAS_SKILL_DIR.parent)}  ({p.stat().st_size} bytes)")

errs, warns = validate_skill(GOTCHAS_SKILL_DIR)
print()
for e in errs:  print("  ERROR  ", e)
for w in warns: print("  WARN   ", w)
print("\n" + ("✓ VALID — conforms to the Agent Skills spec" if not errs
               else f"✕ {len(errs)} error(s) to fix"))

  sarvam-sdk-gotchas/SKILL.md  (5418 bytes)
  name        : sarvam-sdk-gotchas
  description : 507 chars
  body        : 107 lines, ~1177 tokens


✓ VALID — conforms to the Agent Skills spec


---
---
## 7 · The measurement — does any of this actually help?

Same task. Same model. Six different context payloads. Scored against your own
gotchas list.

This is the section that makes the lab worth running. Everything above is a claim;
this is the evidence.

> **Why Context7 isn't in this comparison.** Every layer below is a static string
> prepended to one prompt. Context7 isn't that — it's an MCP tool an agent queries
> *during* its own reasoning loop, which this notebook's single `generate()` call
> doesn't simulate. Measuring it honestly needs a real MCP-wired agent (Lab 09/10's
> pattern), not a bigger context string.

In [19]:
# Build the context payload for each layer
skill_text = (SKILL_DIR / "SKILL.md").read_text(encoding="utf-8")
gotchas_skill_text = (GOTCHAS_SKILL_DIR / "SKILL.md").read_text(encoding="utf-8")

LAYERS = [
    ("0 · no context",              ""),
    ("1a · markdown (chat only)",   mark[:8000] if mark else ""),
    ("1b · markdown (chat+stt)",    ctx_two_pages[:12000] if ctx_two_pages else ""),
    ("2 · llms.txt retrieval",      ctx_llms[:8000] if ctx_llms else ""),
    ("3 · cost-metering skill",     skill_text),
    ("4 · sdk-gotchas skill",       gotchas_skill_text),
    ("5 · gotchas skill + markdown", (gotchas_skill_text + "\n\n" + (mark or ""))[:12000]),
]

runs = []
for label, ctx in LAYERS:
    if not ctx and label != "0 · no context":
        print(f"  skipping {label} — context unavailable")
        continue
    out = generate(TASK, ctx, label=label)
    pts, mx, failed = score(out)
    runs.append({"layer": label, "score": pts, "max": mx,
                 "failed": failed, "ctx_tokens": approx_tokens(ctx), "code": out})
    print(f"     → {pts}/{mx} gotchas avoided   (context ~{approx_tokens(ctx):,} tokens)")

── 0 · no context ── (94 in / 305 out)
     → 16/16 gotchas avoided   (context ~0 tokens)
── 1a · markdown (chat only) ── (2256 in / 255 out)
     → 15/16 gotchas avoided   (context ~2,000 tokens)
── 1b · markdown (chat+stt) ── (3481 in / 207 out)
     → 15/16 gotchas avoided   (context ~3,000 tokens)
── 2 · llms.txt retrieval ── (2489 in / 189 out)
     → 15/16 gotchas avoided   (context ~1,576 tokens)
── 3 · cost-metering skill ── (844 in / 194 out)
     → 15/16 gotchas avoided   (context ~550 tokens)
── 4 · sdk-gotchas skill ── (1706 in / 210 out)
     → 16/16 gotchas avoided   (context ~1,348 tokens)
── 5 · gotchas skill + markdown ── (3431 in / 259 out)
     → 16/16 gotchas avoided   (context ~3,000 tokens)


In [20]:
# ── The result table + a chart you can screenshot ────────────────────────
if runs:
    print(f"{'layer':<24}{'score':>8}{'ctx tokens':>13}   {'bar':<24}")
    print("─" * 72)
    for r in runs:
        bar = "█" * int(r["score"] / r["max"] * 22)
        print(f"{r['layer']:<24}{r['score']:>3}/{r['max']:<4}{r['ctx_tokens']:>13,}   {bar}")
    print("─" * 72)

    best = max(runs, key=lambda r: r["score"])
    base = runs[0]
    print(f"\nbest layer : {best['layer']}  ({best['score']}/{best['max']})")
    print(f"improvement over no context: +{best['score'] - base['score']} gotchas avoided")

    still = set(best["failed"])
    if still:
        print(f"\nStill failing even at the best layer: {sorted(still)}")
        print("Each of those is a candidate line for your NEXT skill revision.")
    else:
        print("\nClean sweep at the best layer.")

layer                      score   ctx tokens   bar                     
────────────────────────────────────────────────────────────────────────
0 · no context           16/16              0   ██████████████████████
1a · markdown (chat only) 15/16          2,000   ████████████████████
1b · markdown (chat+stt) 15/16          3,000   ████████████████████
2 · llms.txt retrieval   15/16          1,576   ████████████████████
3 · cost-metering skill  15/16            550   ████████████████████
4 · sdk-gotchas skill    16/16          1,348   ██████████████████████
5 · gotchas skill + markdown 16/16          3,000   ██████████████████████
────────────────────────────────────────────────────────────────────────

best layer : 0 · no context  (16/16)
improvement over no context: +0 gotchas avoided

Clean sweep at the best layer.


In [21]:
# Show the winning generation so the room can read what good looks like
if runs:
    print(f"── {best['layer']} ──\n")
    print(best["code"][:1600])

── 0 · no context ──


```python
from sarvamai import SarvamAIClient
from sarvamai.audio import AudioTranscriptionRequest
from sarvamai.chat import ChatCompletionRequest
from sarvamai.models import Model

client = SarvamAIClient(api_key="YOUR_SARVAM_API_KEY")

# 1. Transcribe Hindi, 8 kHz telephony audio
transcript_request = AudioTranscriptionRequest(
    file_path="./data/call.wav",
    language="hi-IN",
    mode="transcribe",
    sample_rate=8000,
    audio_format="wav",
    model=Model.SAARIKA_V2
)

transcription = client.audio.transcribe(transcript_request)
text = transcription.text

# 2. Send transcript to Sarvam chat model for a one-line summary
chat_request = ChatCompletionRequest(
    model=Model.SARVAM_M_105B,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Summarize the following Hindi transcript into a single line in English."},
        {"role": "user", "content": text}
    ],
    max_tokens=100
)

response = client.chat.complete(chat_reque

In [22]:
cost.report()

LLM          ₹   0.0176  94 in / 203 out
LLM          ₹   0.2829  9129 in / 213 out
LLM          ₹   0.0251  94 in / 305 out
LLM          ₹   0.0847  2256 in / 255 out
LLM          ₹   0.1171  3481 in / 207 out
LLM          ₹   0.0867  2489 in / 189 out
LLM          ₹   0.0389  844 in / 194 out
LLM          ₹   0.0653  1706 in / 210 out
LLM          ₹   0.1194  3431 in / 259 out
TOTAL        ₹   0.8377
              (₹1000 free credit → ₹999.16 left)
              Estimated from published rates; actual billing usually lower


0.83774472

---
## 8 · Which layer, when

| Situation | Reach for |
|---|---|
| One page, one question, right now | **Markdown** — append `.md`, paste, done |
| "What does this platform even offer?" | **llms.txt** — the map, then fetch the pages that matter |
| Working across many libraries at once | **Context7** — one uniform way to reach all their docs |
| Your team repeats the same mistakes | **Agent Skills** — encode the fix once, everyone inherits it |
| The API ships faster than any index | **llms.txt** — the vendor publishes it; nothing is fresher |
| You need something enforced, not suggested | **Agent Skills** — docs describe, skills prescribe |

**The two-layer default that works for most teams:** an **Agent Skill** carrying your
house rules and the gotchas, plus **`llms.txt`** for live detail. The skill is
prescriptive and yours; `llms.txt` is descriptive and current. Between them you have
covered both halves of the problem.

**The compounding move.** Every time a teammate hits a Sarvam trap, add one line to
the skill. Six months in, that file is the most valuable engineering artefact your
team owns — and unlike documentation, it is loaded automatically by every assistant
your team uses.

---
## ✅ Checkpoint

- [ ] You saw a context-free assistant produce code with real, named defects
- [ ] Your gotchas list runs as an automated grader
- [ ] You measured the token difference between HTML and Markdown docs
- [ ] You parsed `llms.txt` and used it to retrieve only the pages you needed
- [ ] Context7's config is written to disk and you know the Sarvam library ID
- [ ] You **authored and validated** an Agent Skill against the real spec
- [ ] You have a bar chart showing which layer actually helped most

## 🧪 Try this

1. **Add a gotcha.** Next trap you hit, write the regex, add the row, re-run §7.
   Does your best layer still win?
2. **Shrink the skill.** Halve the `SKILL.md` body and re-measure. Where is the
   knee — how little can you say and keep the score?
3. **Rewrite the description only.** Leave the body untouched, sharpen the
   `description` field. Does activation improve? (This is triage-prompt engineering.)
4. **Swap the model.** Run §7 with `sarvam-m` instead of `sarvam-105b`. Does good
   context close the gap between a small model and a large one? Usually: yes, mostly.
5. **Ship it.** Polish the `sarvam-cost-metering` skill and open a PR against
   [`sarvamai/skills`](https://github.com/sarvamai/skills). It is a genuine gap in
   their set, and an evening's work.